# MNIST 手写数字识别（CNN / SVM / 随机森林）

环境：RTX 3050 Laptop / 8GB 内存 / 10 代 i5-H

内容：
1. CNN（PyTorch，约 42 万参数，batch_size=128，10 个 epoch）
2. SVM（scikit-learn，线性核）
3. 随机森林（scikit-learn）

三个模型训练完都会保存到 `models/` 目录，**无需租服务器**。

> 注意：当前安装的 PyTorch 是 CPU 版（见下方 `CUDA 可用: False`）。
> MNIST 很小，CPU 也能在几分钟内跑完；若想用 GPU 加速，需重装 CUDA 版 PyTorch。
> SVM / 随机森林需要额外安装 `scikit-learn` 和 `joblib`：`pip install scikit-learn joblib`。

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

# 让 matplotlib 正常显示中文
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA 可用:', torch.cuda.is_available())

In [ ]:
# 下载并加载 MNIST 数据集
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print('训练集样本数:', len(train_dataset))
print('测试集样本数:', len(test_dataset))
print('单个样本形状:', train_dataset[0][0].shape)

In [ ]:
# 随手看几张训练图片长什么样
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    img, label = train_dataset[i]
    axes[i].imshow(img.squeeze(), cmap='gray')
    axes[i].set_title(str(label))
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 1. 定义 CNN 模型

结构：`Conv(1→32) → Pool` → `Conv(32→64) → Pool` → `FC(3136→128) → Dropout → FC(128→10)`

参数量约 42 万，对你的配置来说非常轻量。

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 28x28 -> 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CNN()
print(model)
print('可训练参数量:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 2. 训练配置

- `BATCH_SIZE = 128`：3050（4GB 显存）上最省心，显存/内存都远远够用
- 优化器 Adam，学习率 1e-3
- Windows 下 `num_workers=0` 最稳，避免多进程报错

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用的设备:', device)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 3. 训练循环

每个 epoch 结束后在测试集上算一次准确率，方便观察。

In [ ]:
EPOCHS = 10
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_loss)

    # 每个 epoch 在测试集上快速评估
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
    acc = correct / len(test_loader.dataset)

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | loss: {epoch_loss:.4f} | 测试准确率: {acc*100:.2f}%')

In [ ]:
# 训练损失曲线
plt.figure(figsize=(6, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss Curve')
plt.grid(True)
plt.show()

## 4. 最终评估与预测可视化

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f'最终测试集准确率: {correct / total * 100:.2f}%')

In [ ]:
# 可视化一批预测结果（绿色=预测正确，红色=预测错误）
model.eval()
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap='gray')
    color = 'green' if preds[i] == labels[i] else 'red'
    ax.set_title(f'预测 {preds[i].item()} / 真 {labels[i].item()}', color=color, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. 保存训练好的 CNN

保存 `state_dict`（只存权重，约 1.7MB），比整模型序列化更推荐。
文件会存到当前 notebook 目录下的 `models/mnist_cnn.pth`。

In [ ]:
import os

os.makedirs('models', exist_ok=True)
save_path = 'models/mnist_cnn.pth'
torch.save(model.state_dict(), save_path)

print('CNN 模型已保存到:', os.path.abspath(save_path))
print('文件大小: %.2f MB' % (os.path.getsize(save_path) / 1024 / 1024))

## 6. 之后如何加载 CNN

**先要有同样的模型结构**（运行第 1 节的 `CNN` 定义），再加载权重。
`map_location=device` 让它在没有 GPU 的机器上也能加载。

In [ ]:
loaded_model = CNN()
loaded_model.load_state_dict(torch.load(save_path, map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()

print('CNN 模型加载完成，可用于推理')

## 7. 传统机器学习：准备展平数据

SVM 和随机森林需要 784 维的一维向量，而不是 28×28 图像。
这里直接把原始像素缩放到 0~1（不用 torch 的归一化），两个模型共用这份数据。

In [ ]:
import os

# 用 MNIST 原始像素（uint8, 0~255）拉平并缩放到 0~1
X_train = train_dataset.data.float().numpy().reshape(-1, 28 * 28) / 255.0
y_train = train_dataset.targets.numpy()

X_test = test_dataset.data.float().numpy().reshape(-1, 28 * 28) / 255.0
y_test = test_dataset.targets.numpy()

print('X_train:', X_train.shape, '| y_train:', y_train.shape)
print('X_test :', X_test.shape, '| y_test :', y_test.shape)

## 8. SVM（线性核）

RBF 核在 6 万样本上训练非常慢（可能几十分钟），这里用 `LinearSVC`，几十秒完成，准确率约 90% 左右。
模型用 `joblib` 保存到 `models/svm.joblib`。

In [ ]:
import joblib
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

svm = LinearSVC(C=1.0, max_iter=2000)
svm.fit(X_train, y_train)

svm_acc = accuracy_score(y_test, svm.predict(X_test))
print(f'SVM 测试集准确率: {svm_acc * 100:.2f}%')

svm_path = 'models/svm.joblib'
joblib.dump(svm, svm_path)
print('SVM 已保存到:', os.path.abspath(svm_path))

## 9. 随机森林

100 棵树，`n_jobs=-1` 用满所有 CPU 核，训练约 1~3 分钟，准确率约 96%~97%。
模型保存到 `models/random_forest.joblib`。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f'随机森林测试集准确率: {rf_acc * 100:.2f}%')

rf_path = 'models/random_forest.joblib'
joblib.dump(rf, rf_path)
print('随机森林已保存到:', os.path.abspath(rf_path))

## 10. 之后如何加载 SVM / 随机森林

`joblib.load` 直接还原模型，不需要重新定义结构，比 PyTorch 更省事。

In [ ]:
svm_loaded = joblib.load(svm_path)
rf_loaded = joblib.load(rf_path)

print('SVM 加载完成，准确率: %.2f%%' % (accuracy_score(y_test, svm_loaded.predict(X_test)) * 100))
print('随机森林加载完成，准确率: %.2f%%' % (accuracy_score(y_test, rf_loaded.predict(X_test)) * 100))